In [3]:
import os
import json
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
from tqdm import tqdm
import time

In [ ]:
import torchvision.models as models

class ShadowDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Pretrained ResNet18 backbone
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        
        # Remove final classification layer
        self.backbone.fc = nn.Identity()
        
        # Regression head (8 values = 4 corners)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 8)  # 4 corners (x,y)
        )

        self.dir_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        corners = self.head(features)
        direction = self.dir_head(features)
        return corners, direction  # keep your interface consistent

In [5]:
class ShadowDataset(Dataset):
    def __init__(self, folder, img_size=224):
        self.folder = folder
        self.img_size = img_size
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                        [0.229, 0.224, 0.225])   # ImageNet std
        ])
        
        # Collect all images that have a matching json
        self.samples = [
            f.replace(".png", "")
            for f in os.listdir(folder)
            if f.endswith(".png") and 
               os.path.exists(os.path.join(folder, f.replace(".png", ".json")))
        ]
        print(f"Found {len(self.samples)} samples in {folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        
        # ── Load image ──
        img_path = os.path.join(self.folder, f"{name}.png")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size          # original size, needed for normalizing
        img = self.transform(img)
        
        # ── Load annotation ──
        json_path = os.path.join(self.folder, f"{name}.json")
        with open(json_path) as f:
            ann = json.load(f)
        
        # Normalize all corner coordinates to 0-1
        # Note: values CAN be > 1.0 since person is off-screen!
        bbox = ann["bbox"]
        corners = torch.tensor([
            bbox["top_left"][0]     / W,
            bbox["top_left"][1]     / H,
            bbox["top_right"][0]    / W,
            bbox["top_right"][1]    / H,
            bbox["bottom_left"][0]  / W,
            bbox["bottom_left"][1]  / H,
            bbox["bottom_right"][0] / W,
            bbox["bottom_right"][1] / H,
        ], dtype=torch.float32)

        direction = torch.tensor(
            [ann["walking_into_frame_bool"]], 
        dtype=torch.float32
)
        
        return img, corners, direction, W, H, name


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Load dataset
dataset = ShadowDataset("data/train_data/train_data")

# Split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=32)

# Model
model = ShadowDetector().to(device)

# Loss + optimizer
criterion_bbox = nn.SmoothL1Loss()
criterion_dir = nn.BCEWithLogitsLoss()  # for direction prediction
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float("inf")
THRESHOLD = 0.70

for epoch in range(50):
    start = time.time()

    # Freeze backbone for first 10 epochs
    for param in model.backbone.parameters():
        param.requires_grad = epoch >= 10

    # ── TRAIN ──
    model.train()
    train_bbox_losses, train_dir_losses = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/50 [Train]", leave=False)
    for imgs, corners, direction, _, _, _ in loop:
        imgs    = imgs.to(device)
        corners = corners.to(device)
        direction = direction.to(device)
        optimizer.zero_grad()

        pred_corners, pred_dir = model(imgs)   # ignore direction output

        bbox_loss = criterion_bbox(pred_corners, corners)
        dir_loss  = criterion_dir(pred_dir.squeeze(1), direction.squeeze(1))
        loss      = bbox_loss + 0.1 * dir_loss

        loss.backward()
        optimizer.step()

        train_bbox_losses.append(bbox_loss.item())
        train_dir_losses.append(dir_loss.item())
        loop.set_postfix(bbox=f"{bbox_loss.item():.4f}", dir=f"{dir_loss.item():.4f}")

    # ── VALIDATE ──
    model.eval()
    val_bbox_losses, val_dir_losses = [], []
    all_probs, all_labels = [], []

    loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/50 [Val]", leave=False)
    with torch.no_grad():
        for imgs, corners, direction, _, _, _ in loop:
            imgs      = imgs.to(device)
            corners   = corners.to(device)
            direction = direction.to(device)

            pred_corners, pred_dir = model(imgs)

            bbox_loss = criterion_bbox(pred_corners, corners)
            dir_loss  = criterion_dir(pred_dir.squeeze(1), direction.squeeze(1))

            val_bbox_losses.append(bbox_loss.item())
            val_dir_losses.append(dir_loss.item())

            probs = torch.sigmoid(pred_dir.squeeze(1))
            all_probs.append(probs.cpu())
            all_labels.append(direction.squeeze(1).cpu())

            loop.set_postfix(bbox=f"{bbox_loss.item():.4f}", dir=f"{dir_loss.item():.4f}")

    # ── METRICS ──
    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)

    confident_mask = (all_probs > THRESHOLD) | (all_probs < 1 - THRESHOLD)
    coverage = confident_mask.float().mean()
    acc_when_confident = (
        (all_probs[confident_mask] > 0.5) == all_labels[confident_mask].bool()
    ).float().mean() if confident_mask.any() else torch.tensor(0.0)

    avg_train_bbox = sum(train_bbox_losses) / len(train_bbox_losses)
    avg_val_bbox   = sum(val_bbox_losses)   / len(val_bbox_losses)
    elapsed        = time.time() - start

    print(
        f"Epoch {epoch+1:02d}/50 | "
        f"Train bbox: {avg_train_bbox:.4f} | Val bbox: {avg_val_bbox:.4f} | "
        f"Coverage: {coverage:.1%} | Dir acc: {acc_when_confident:.1%} | "
        f"Time: {elapsed:.1f}s"
    )

    if avg_val_bbox < best_val_loss:
        best_val_loss = avg_val_bbox
        torch.save(model.state_dict(), "best_model.pth")
        print("↑ Saved best model")

    scheduler.step()

In [ ]:
# Load best model
model = ShadowDetector().to(device)
model.load_state_dict(torch.load("best_model.pth"))

def predict(model, img, threshold=0.70):
    model.eval()
    with torch.no_grad():
        pred_corners, pred_dir_logit = model(img)

        prob = torch.sigmoid(pred_dir_logit).item()

        if prob > threshold:
            direction = 1      # walking into frame
        elif prob < (1 - threshold):
            direction = 0      # walking away
        else:
            direction = -1     # not confident enough

    return pred_corners, direction, prob

# Example: run on a single image
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

img = Image.open("path/to/your/image.png").convert("RGB")
img_tensor = transform(img).unsqueeze(0).to(device)

corners, direction, prob = predict(model, img_tensor)

print(f"Corners: {corners}")
print(f"Direction: {direction} (prob: {prob:.2f})")